# 실습 문제 모범답안 - 401. Deep Agent 고급 기능

`401_Deep_Agent_Advanced` 노트북 실습 문제(이메일 발송 승인 워크플로우)에 대한 모범답안 예시입니다.

### 구현 요약

1. `send_email` 스텁 도구 정의
2. `interrupt_on={"send_email": True}` + 체크포인터로 에이전트 생성
3. 메일 발송 요청 → `__interrupt__`의 `action_requests` 확인
4. **reject** 결정으로 재개 → 에이전트가 거부 사유를 반영해 응답

**기대 결과**: 도구는 실행되지 않고, 에이전트가 거부 사유("지금은 보내지 말 것")를
받아들여 메일을 보내지 않았다고 안내합니다.

In [1]:
from dotenv import load_dotenv

load_dotenv()

True

## 1. 도구 정의 + 에이전트 생성

본문의 `delete_file` 예제와 같은 구조입니다. HITL에는 체크포인터가 필수입니다.

In [2]:
import uuid
from langchain.tools import tool
from langgraph.checkpoint.memory import InMemorySaver
from deepagents import create_deep_agent

@tool
def send_email(to: str, subject: str) -> str:
    """지정한 수신자에게 이메일을 발송합니다."""
    return f"'{to}'에게 '{subject}' 메일을 발송했습니다. (시뮬레이션)"


email_agent = create_deep_agent(
    model="openai:gpt-5-mini",
    tools=[send_email],
    system_prompt=(
        "당신은 업무 비서입니다. "
        "사용자가 메일 발송을 요청하면 추가 확인 질문 없이 send_email 도구를 즉시 호출하세요. "
        "실행 승인 여부는 시스템(Human-in-the-Loop)이 처리합니다."
    ),
    checkpointer=InMemorySaver(),        # HITL에는 체크포인터가 필수
    interrupt_on={"send_email": True},   # 이 도구 실행 전에 일시정지
)

config = {"configurable": {"thread_id": str(uuid.uuid4())}}

C:\Users\trimu\anaconda3\envs\langchain\Lib\site-packages\langgraph\checkpoint\base\__init__.py:17: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


## 2. 메일 발송 요청 → 인터럽트 확인

In [3]:
result = email_agent.invoke(
    {"messages": [{"role": "user", "content": "팀장님(boss@company.com)께 '주간 보고' 제목으로 메일을 보내줘."}]},
    config,
)

if "__interrupt__" in result:
    interrupt_value = result["__interrupt__"][0].value

    print("실행이 일시정지되었습니다!\n")
    print("승인 대기 중인 작업 (action_requests):")
    for req in interrupt_value["action_requests"]:
        print(f"  - 도구: {req['name']}, 인자: {req['args']}")
else:
    print("인터럽트가 발생하지 않았습니다.")

실행이 일시정지되었습니다!

승인 대기 중인 작업 (action_requests):
  - 도구: send_email, 인자: {'to': 'boss@company.com', 'subject': '주간 보고'}


## 3. 거부(reject) 후 재개

`reject`는 도구를 실행하지 않고, `message`에 적은 거부 사유를 에이전트에게 전달합니다.
에이전트는 그 사유를 보고 다음 행동을 결정합니다.

In [4]:
from langgraph.types import Command

resumed = email_agent.invoke(
    Command(resume={"decisions": [{
        "type": "reject",
        "message": "지금은 보내지 마세요. 보고 내용 검토 후 내일 다시 요청하겠습니다.",
    }]}),
    config,  # 동일한 thread_id
)

resumed["messages"][-1].pretty_print()

================================== Ai Message ==================================

[{'id': 'rs_0af3d5f7a74d4441006a4a4bf65c8c819ab592c57d918c46bf', 'summary': [], 'type': 'reasoning', 'content': []}, {'type': 'text', 'text': '요청을 접수했습니다 — 수신: boss@company.com, 제목: "주간 보고".\n\n시스템 응답: "지금은 보내지 마세요. 보고 내용 검토 후 내일 다시 요청하겠습니다."\n\n계속 진행할지(예: 내일 자동으로 다시 요청하거나, 메일 내용 작성/첨부 등) 지시해 주세요.', 'annotations': [], 'id': 'msg_0af3d5f7a74d4441006a4a4bf9ccf0819aa35802531212fd51'}]


## 확인 포인트

- 인터럽트 시점에 `send_email`은 **아직 실행되지 않은 상태**입니다 — 사람이 결정하기 전까지 도구가 호출되지 않습니다.
- `reject`로 재개하면 도구는 끝내 실행되지 않고, 에이전트는 거부 사유를 받아 "메일을 보내지 않았다"고 안내합니다.
- 본문의 `approve` 흐름과 비교해 보세요: 같은 인터럽트에서 사람의 결정에 따라 실행 여부가 갈립니다.